In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

## New material

### How can we predict the outcome of a new individual?

In [ ]:
families = Table.read_table('family_heights.csv')
families

Note: Child heights are the **adult** heights of children in a family

In [ ]:
families.sort('family', descending=True).show(6)

In [ ]:
families.where('family', are.equal_to('5'))

We will now average the heights of the parents and create a table with two columns:

- Parent average height
- Child height

In [ ]:
parent_avgs = (families.column('father') + families.column('mother'))/2

In [ ]:
heights = Table().with_columns('Parent Average', parent_avgs, 
                               'Child', families.column('child'))
heights

**STOP**

### Looking for association between two numerical variables

**Discussion Question (2x)**

In [ ]:
heights.scatter('Parent Average', 'Child')

In [ ]:
nearby = heights.where('Parent Average', are.between(67.5, 68.5))
nearby_mean = np.average(nearby.column('Child'))
nearby_mean

In [ ]:
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2)
plots.scatter(68, nearby_mean, color='red', s=50);

In [ ]:
def predict_child(h):
    """Predict the height of a child whose parents have a parent average height of p_avg.
    
    The prediction is the average height of the children whose parent average height is
    in the range p_avg plus or minus 0.5.
    """
    nearby = heights.where('Parent Average', are.between(h - 1/2, h + 1/2))
    return np.average(nearby.column('Child'))

In [ ]:
heights_with_predictions = heights.with_columns(
    'Prediction', heights.apply(predict_child, 'Parent Average'))

In [ ]:
heights_with_predictions.scatter('Parent Average')

#### More associations!

In [ ]:
hybrid = Table.read_table('hybrid.csv')

In [ ]:
hybrid.sample(6)

In [ ]:
hybrid.scatter('acceleration', 'msrp')
plots.title('MSRP is positively associated with acceleration');

In [ ]:
suv = hybrid.where('class', 'SUV')
suv.num_rows

In [ ]:
suv.scatter('acceleration', 'msrp')
plots.title('This relationship also holds with SUVs!');

In [ ]:
hybrid.scatter('mpg', 'msrp')
plots.title('This is a wacky relationship!');

In [ ]:
suv.scatter('mpg', 'msrp')
plots.title("MSRP is more clearly linearly negatively associated with MPG \n when just considering SUVs");

**STOP**

### $r$ helps us measure linear association

In [ ]:
def standard_units(x):
    "Convert any array of numbers to standard units."
    return (x - np.average(x)) / np.std(x)

In [ ]:
def correlation(t, x, y):
    """t is a table; x and y are column labels"""
    x_in_standard_units = standard_units(t.column(x))
    y_in_standard_units = standard_units(t.column(y))
    return np.average(x_in_standard_units * y_in_standard_units)

***Task***: Use the `r_scatter` function defined below to see how changing $r$ changes a scatter plot between numerical $x$ and $y$.`

In [ ]:
def r_scatter(r):
    plots.figure(figsize=(5,5))
    "Generate a scatter plot with a correlation approximately r"
    x = np.random.normal(0, 1, 1000)
    z = np.random.normal(0, 1, 1000)
    y = r*x + (np.sqrt(1-r**2))*z
    plots.scatter(x, y, color='darkblue', s=20)
    plots.xlim(-4, 4)
    plots.ylim(-4, 4)

In [ ]:
r_scatter(1)

In [ ]:
correlation(hybrid, 'acceleration', 'msrp')

In [ ]:
correlation(suv, 'mpg', 'msrp')

### Here’s a more technical definition of $r$!


In [ ]:
x = np.arange(1, 7, 1)
y = make_array(2, 3, 1, 5, 2, 7)
t = Table().with_columns(
        'x', x,
        'y', y
    )
t

In [ ]:
t.scatter('x', 'y', s=30, color='red')

In [ ]:
t = t.with_columns(
        'x (standard units)', standard_units(x),
        'y (standard units)', standard_units(y)
    )
t

In [ ]:
t = t.with_columns(
    'product of standard units', t.column(3) * t.column(2))
t

In [ ]:
np.average(t.column('product of standard units'))

In [ ]:
def correlation(t, x, y):
    """t is a table; x and y are column labels"""
    x_in_standard_units = standard_units(t.column(x))
    y_in_standard_units = standard_units(t.column(y))
    return np.average(x_in_standard_units * y_in_standard_units)

In [ ]:
correlation(t, 'x', 'y')

**Discussion Question**

In [ ]:
correlation(t, 'y', 'x')

### It's always good to pair a calculation of $r$ with a visualization.

In [ ]:
new_x = np.arange(-4, 4.1, 0.5)
nonlinear = Table().with_columns(
        'x', new_x,
        'y', new_x**2
    )
nonlinear.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(nonlinear, 'x', 'y')

In [ ]:
nonlinear.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(hybrid, 'mpg', 'msrp')

In [ ]:
hybrid.scatter('mpg', 'msrp')
plots.title('This is a wacky relationship!');

**STOP**

### Be careful with how you interpret $r$!

#### Outliers can greatly affect the value of $r$

In [ ]:
line = Table().with_columns(
        'x', make_array(1, 2, 3, 4),
        'y', make_array(1, 2, 3, 4)
    )
line.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(line, 'x', 'y')

In [ ]:
line = Table().with_columns(
        'x', make_array(1, 2, 3, 100),
        'y', make_array(1, 2, 3, 100)
    )
line.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(line, 'x', 'y')

____

In [ ]:
outlier = Table().with_columns(
        'x', make_array(1, 2, 3, 4, 5),
        'y', make_array(1, 2, 3, 4, 0)
    )
outlier.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(outlier, 'x', 'y')

#### Correlations based on aggregated data can be misleading

In [ ]:
sat2014 = Table.read_table('sat_scores.csv').sort('State')
sat2014

In [ ]:
correlation(sat2014, 'Critical Reading', 'Math')

In [ ]:
sat2014.scatter('Critical Reading', 'Math')

**Discussion Question**

In [ ]:
def rate_code(x):
    if x <= 25:
        return 'low'
    elif x <= 50:
        return 'low-moderate'
    elif x <= 75:
        return 'moderate_high'
    else:
        return 'high'

In [ ]:
rate_codes = sat2014.apply(rate_code, 'Participation Rate')

In [ ]:
sat2014 = sat2014.with_columns('Rate Code', rate_codes)
sat2014

In [ ]:
sat2014.scatter('Critical Reading', 'Math', group='Rate Code')
plots.title("Each individual point is a state, not a student!");